In [2]:
import pandas as pd
import numpy as np
from scipy import stats

# ---------------------------------------------------------
# LOAD CLEANED EXPERIMENT DATA
# ---------------------------------------------------------

input_file = "Cleaned_Growth_Experiment.xlsx"

df = pd.read_excel(
    input_file,
    sheet_name="Experiment Data"
)

# Basic formatting
df["experiment_date"] = pd.to_datetime(
    df["experiment_date"],
    errors="coerce"
)

# ---------------------------------------------------------
# BASIC OVERVIEW
# ---------------------------------------------------------

total_users = df["student_id"].nunique()
total_records = len(df)

control = df[df["variant"] == "Control"].copy()
treatment = df[df["variant"] == "Treatment"].copy()

control_users = len(control)
treatment_users = len(treatment)

control_conversions = control["primary_conversion"].sum()
treatment_conversions = treatment["primary_conversion"].sum()

control_rate = (
    control_conversions / control_users
    if control_users > 0 else 0
)

treatment_rate = (
    treatment_conversions / treatment_users
    if treatment_users > 0 else 0
)

# ---------------------------------------------------------
# EFFECT CALCULATION
# ---------------------------------------------------------

absolute_effect = treatment_rate - control_rate

relative_effect = (
    absolute_effect / control_rate
    if control_rate > 0 else np.nan
)

# ---------------------------------------------------------
# 95% CONFIDENCE INTERVAL
# Difference in proportions
# ---------------------------------------------------------

se = np.sqrt(
    (control_rate * (1 - control_rate) / control_users)
    +
    (treatment_rate * (1 - treatment_rate) / treatment_users)
)

z_value = 1.96

ci_lower = absolute_effect - z_value * se
ci_upper = absolute_effect + z_value * se

# ---------------------------------------------------------
# STATISTICAL SIGNIFICANCE
# ---------------------------------------------------------

successes = np.array([
    treatment_conversions,
    control_conversions
])

failures = np.array([
    treatment_users - treatment_conversions,
    control_users - control_conversions
])

contingency_table = np.array([
    successes,
    failures
])

chi2_stat, p_value, dof, expected = stats.chi2_contingency(
    contingency_table
)

statistically_significant = (
    p_value < 0.05
)

# ---------------------------------------------------------
# SRM CHECK
# ---------------------------------------------------------

observed_counts = np.array([
    control_users,
    treatment_users
])

expected_counts = np.array([
    total_users * 0.50,
    total_users * 0.50
])

srm_stat, srm_p_value = stats.chisquare(
    observed_counts,
    f_exp=expected_counts
)

srm_pass = srm_p_value >= 0.05

# ---------------------------------------------------------
# GUARDRAIL METRICS
# ---------------------------------------------------------

control_engagement = control["guardrail_engagement"].mean()
treatment_engagement = treatment["guardrail_engagement"].mean()

engagement_effect = (
    treatment_engagement - control_engagement
)

control_search_time = (
    control["search_time_guardrail"].mean()
)

treatment_search_time = (
    treatment["search_time_guardrail"].mean()
)

search_time_change = (
    treatment_search_time - control_search_time
)

control_processing_time = (
    control["processing_time_guardrail"].mean()
)

treatment_processing_time = (
    treatment["processing_time_guardrail"].mean()
)

processing_time_change = (
    treatment_processing_time - control_processing_time
)

# Guardrail rules
engagement_guardrail_pass = (
    treatment_engagement >= control_engagement * 0.95
)

search_guardrail_pass = (
    treatment_search_time <= control_search_time * 1.10
)

processing_guardrail_pass = (
    treatment_processing_time <= control_processing_time * 1.10
)

all_guardrails_pass = (
    engagement_guardrail_pass
    and search_guardrail_pass
    and processing_guardrail_pass
)

# ---------------------------------------------------------
# EXPERIMENT VERDICT
# ---------------------------------------------------------

if (
    statistically_significant
    and absolute_effect > 0
    and srm_pass
    and all_guardrails_pass
):
    verdict = "SHIP"

elif (
    absolute_effect <= 0
    or not all_guardrails_pass
):
    verdict = "NO-SHIP"

else:
    verdict = "INCONCLUSIVE"

# ---------------------------------------------------------
# BUSINESS RECOMMENDATION
# ---------------------------------------------------------

if verdict == "SHIP":
    recommendation = (
        "Treatment shows a positive and statistically significant "
        "primary conversion effect, SRM passes, and guardrails remain "
        "within acceptable limits. Recommend shipping the treatment."
    )

elif verdict == "NO-SHIP":
    recommendation = (
        "The treatment does not provide sufficient evidence of a safe "
        "positive improvement. Recommend not shipping and investigate "
        "the primary metric and guardrail results."
    )

else:
    recommendation = (
        "The evidence is not strong enough for a confident decision. "
        "Recommend collecting more data or running a better-powered "
        "experiment before making the final decision."
    )

# ---------------------------------------------------------
# EXPERIMENT OVERVIEW
# ---------------------------------------------------------

experiment_overview = pd.DataFrame({
    "Metric": [
        "Experiment ID",
        "Total Users",
        "Control Users",
        "Treatment Users",
        "Control Conversions",
        "Treatment Conversions",
        "Control Conversion Rate",
        "Treatment Conversion Rate",
        "Absolute Effect",
        "Relative Effect",
        "P-Value",
        "Statistical Significance",
        "SRM P-Value",
        "SRM Check",
        "Experiment Verdict"
    ],
    "Value": [
        df["experiment_id"].iloc[0],
        total_users,
        control_users,
        treatment_users,
        control_conversions,
        treatment_conversions,
        control_rate,
        treatment_rate,
        absolute_effect,
        relative_effect,
        p_value,
        "Yes" if statistically_significant else "No",
        srm_p_value,
        "Pass" if srm_pass else "Fail",
        verdict
    ]
})

# ---------------------------------------------------------
# CONTROL VS TREATMENT
# ---------------------------------------------------------

control_treatment = pd.DataFrame({
    "Variant": [
        "Control",
        "Treatment"
    ],
    "Users": [
        control_users,
        treatment_users
    ],
    "Conversions": [
        control_conversions,
        treatment_conversions
    ],
    "Conversion Rate": [
        control_rate,
        treatment_rate
    ]
})

# ---------------------------------------------------------
# EFFECT AND CONFIDENCE INTERVAL
# ---------------------------------------------------------

effect_analysis = pd.DataFrame({
    "Metric": [
        "Control Conversion Rate",
        "Treatment Conversion Rate",
        "Absolute Effect",
        "Relative Effect",
        "95% CI Lower",
        "95% CI Upper",
        "P-Value"
    ],
    "Value": [
        control_rate,
        treatment_rate,
        absolute_effect,
        relative_effect,
        ci_lower,
        ci_upper,
        p_value
    ]
})

# ---------------------------------------------------------
# SRM ANALYSIS
# ---------------------------------------------------------

srm_analysis = pd.DataFrame({
    "Variant": [
        "Control",
        "Treatment"
    ],
    "Observed Users": [
        control_users,
        treatment_users
    ],
    "Expected Users": [
        expected_counts[0],
        expected_counts[1]
    ],
    "Observed Share": [
        control_users / total_users,
        treatment_users / total_users
    ],
    "Expected Share": [
        0.50,
        0.50
    ]
})

srm_summary = pd.DataFrame({
    "Metric": [
        "SRM Chi-Square Statistic",
        "SRM P-Value",
        "SRM Result"
    ],
    "Value": [
        srm_stat,
        srm_p_value,
        "Pass" if srm_pass else "Fail"
    ]
})

# ---------------------------------------------------------
# GUARDRAIL ANALYSIS
# ---------------------------------------------------------

guardrail_analysis = pd.DataFrame({
    "Guardrail Metric": [
        "Engagement Rate",
        "Average Search Time",
        "Average Processing Time"
    ],
    "Control": [
        control_engagement,
        control_search_time,
        control_processing_time
    ],
    "Treatment": [
        treatment_engagement,
        treatment_search_time,
        treatment_processing_time
    ],
    "Change": [
        engagement_effect,
        search_time_change,
        processing_time_change
    ],
    "Result": [
        "Pass" if engagement_guardrail_pass else "Fail",
        "Pass" if search_guardrail_pass else "Fail",
        "Pass" if processing_guardrail_pass else "Fail"
    ]
})

# ---------------------------------------------------------
# SEGMENT ANALYSIS - REGION
# ---------------------------------------------------------

region_analysis = (
    df.groupby(["region", "variant"])
    .agg(
        Users=("student_id", "nunique"),
        Conversions=("primary_conversion", "sum"),
        Conversion_Rate=("primary_conversion", "mean")
    )
    .reset_index()
)

region_pivot = region_analysis.pivot(
    index="region",
    columns="variant",
    values="Conversion_Rate"
).reset_index()

if "Control" in region_pivot.columns:
    region_pivot["Control"] = region_pivot["Control"].fillna(0)

if "Treatment" in region_pivot.columns:
    region_pivot["Treatment"] = region_pivot["Treatment"].fillna(0)

if (
    "Control" in region_pivot.columns
    and "Treatment" in region_pivot.columns
):
    region_pivot["Effect"] = (
        region_pivot["Treatment"]
        - region_pivot["Control"]
    )

# ---------------------------------------------------------
# SEGMENT ANALYSIS - SKILL
# ---------------------------------------------------------

skill_analysis = (
    df.groupby(["skill", "variant"])
    .agg(
        Users=("student_id", "nunique"),
        Conversions=("primary_conversion", "sum"),
        Conversion_Rate=("primary_conversion", "mean")
    )
    .reset_index()
)

skill_pivot = skill_analysis.pivot(
    index="skill",
    columns="variant",
    values="Conversion_Rate"
).reset_index()

if "Control" in skill_pivot.columns:
    skill_pivot["Control"] = skill_pivot["Control"].fillna(0)

if "Treatment" in skill_pivot.columns:
    skill_pivot["Treatment"] = skill_pivot["Treatment"].fillna(0)

if (
    "Control" in skill_pivot.columns
    and "Treatment" in skill_pivot.columns
):
    skill_pivot["Effect"] = (
        skill_pivot["Treatment"]
        - skill_pivot["Control"]
    )

# ---------------------------------------------------------
# SEGMENT ANALYSIS - EXPERIENCE
# ---------------------------------------------------------

df["experience_group"] = pd.cut(
    df["experience_years"],
    bins=[-1, 1, 3, 5, np.inf],
    labels=[
        "0-1 Years",
        "2-3 Years",
        "4-5 Years",
        "5+ Years"
    ]
)

experience_analysis = (
    df.groupby(
        ["experience_group", "variant"],
        observed=False
    )
    .agg(
        Users=("student_id", "nunique"),
        Conversions=("primary_conversion", "sum"),
        Conversion_Rate=("primary_conversion", "mean")
    )
    .reset_index()
)

experience_pivot = experience_analysis.pivot(
    index="experience_group",
    columns="variant",
    values="Conversion_Rate"
).reset_index()

if "Control" in experience_pivot.columns:
    experience_pivot["Control"] = (
        experience_pivot["Control"].fillna(0)
    )

if "Treatment" in experience_pivot.columns:
    experience_pivot["Treatment"] = (
        experience_pivot["Treatment"].fillna(0)
    )

if (
    "Control" in experience_pivot.columns
    and "Treatment" in experience_pivot.columns
):
    experience_pivot["Effect"] = (
        experience_pivot["Treatment"]
        - experience_pivot["Control"]
    )

# ---------------------------------------------------------
# MONTHLY ANALYSIS
# ---------------------------------------------------------

df["month"] = df["experiment_date"].dt.to_period(
    "M"
).astype(str)

monthly_analysis = (
    df.groupby(["month", "variant"])
    .agg(
        Users=("student_id", "nunique"),
        Conversions=("primary_conversion", "sum"),
        Conversion_Rate=("primary_conversion", "mean")
    )
    .reset_index()
)

monthly_pivot = monthly_analysis.pivot(
    index="month",
    columns="variant",
    values="Conversion_Rate"
).reset_index()

# ---------------------------------------------------------
# DATA QUALITY
# ---------------------------------------------------------

data_quality = pd.DataFrame({
    "Check": [
        "Total Rows",
        "Duplicate Rows",
        "Missing Values",
        "Unique Students",
        "Unique Companies",
        "Unique Jobs",
        "Control Users",
        "Treatment Users"
    ],
    "Result": [
        len(df),
        df.duplicated().sum(),
        df.isna().sum().sum(),
        df["student_id"].nunique(),
        df["company_id"].nunique(),
        df["job_id"].nunique(),
        control_users,
        treatment_users
    ]
})

# ---------------------------------------------------------
# EXPERIMENT DECISION
# ---------------------------------------------------------

decision_analysis = pd.DataFrame({
    "Decision Area": [
        "Primary Metric",
        "Effect",
        "95% Confidence Interval",
        "Statistical Significance",
        "SRM",
        "Guardrails",
        "Final Verdict",
        "Recommendation"
    ],
    "Result": [
        "Primary Conversion Rate",
        absolute_effect,
        f"{ci_lower:.4f} to {ci_upper:.4f}",
        "Pass" if statistically_significant else "Fail",
        "Pass" if srm_pass else "Fail",
        "Pass" if all_guardrails_pass else "Fail",
        verdict,
        recommendation
    ]
})

# ---------------------------------------------------------
# EXPERIMENT LOG / LEARNING
# ---------------------------------------------------------

experiment_log = pd.DataFrame({
    "Experiment Field": [
        "Experiment ID",
        "Primary Metric",
        "Control",
        "Treatment",
        "Decision Rule",
        "Observed Effect",
        "Statistical Result",
        "SRM Result",
        "Guardrail Result",
        "Final Decision",
        "Business Learning"
    ],
    "Details": [
        df["experiment_id"].iloc[0],
        "Primary Conversion Rate",
        "Control Variant",
        "Treatment Variant",
        "Ship only when positive effect, valid randomization, "
        "statistical evidence, and guardrails pass.",
        f"{absolute_effect:.4f}",
        "Statistically Significant"
        if statistically_significant
        else "Not Statistically Significant",
        "Pass" if srm_pass else "Fail",
        "Pass" if all_guardrails_pass else "Fail",
        verdict,
        (
            "Treatment should be adopted only when the primary "
            "metric improves without unacceptable guardrail impact."
        )
    ]
})

# ---------------------------------------------------------
# BUSINESS INSIGHTS
# ---------------------------------------------------------

business_insights = pd.DataFrame({
    "Insight": [
        "Primary Metric",
        "Treatment Performance",
        "Effect",
        "Confidence Interval",
        "Statistical Evidence",
        "Randomization Quality",
        "Guardrail Health",
        "Final Decision"
    ],
    "Finding": [
        "Primary Conversion Rate",
        f"Treatment conversion rate = {treatment_rate:.2%}",
        f"Absolute effect = {absolute_effect:.2%}",
        f"95% CI = {ci_lower:.2%} to {ci_upper:.2%}",
        (
            "Significant"
            if statistically_significant
            else "Not significant"
        ),
        (
            "SRM Passed"
            if srm_pass
            else "SRM Failed"
        ),
        (
            "All guardrails passed"
            if all_guardrails_pass
            else "One or more guardrails failed"
        ),
        verdict
    ]
})

# ---------------------------------------------------------
# SAVE ONE FINAL PYTHON ANALYSIS RESULT
# ---------------------------------------------------------

output_file = "Python Analysis Result.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    experiment_overview.to_excel(
        writer,
        sheet_name="Experiment Overview",
        index=False
    )

    control_treatment.to_excel(
        writer,
        sheet_name="Control vs Treatment",
        index=False
    )

    effect_analysis.to_excel(
        writer,
        sheet_name="Effect and CI",
        index=False
    )

    srm_analysis.to_excel(
        writer,
        sheet_name="SRM Analysis",
        index=False
    )

    srm_summary.to_excel(
        writer,
        sheet_name="SRM Summary",
        index=False
    )

    guardrail_analysis.to_excel(
        writer,
        sheet_name="Guardrails",
        index=False
    )

    region_pivot.to_excel(
        writer,
        sheet_name="Region Analysis",
        index=False
    )

    skill_pivot.to_excel(
        writer,
        sheet_name="Skill Analysis",
        index=False
    )

    experience_pivot.to_excel(
        writer,
        sheet_name="Experience Analysis",
        index=False
    )

    monthly_pivot.to_excel(
        writer,
        sheet_name="Monthly Analysis",
        index=False
    )

    data_quality.to_excel(
        writer,
        sheet_name="Data Quality",
        index=False
    )

    decision_analysis.to_excel(
        writer,
        sheet_name="Experiment Decision",
        index=False
    )

    experiment_log.to_excel(
        writer,
        sheet_name="Experiment Log",
        index=False
    )

    business_insights.to_excel(
        writer,
        sheet_name="Business Insights",
        index=False
    )

print("======================================")
print("PYTHON ANALYSIS COMPLETED SUCCESSFULLY")
print("======================================")
print("Output File:", output_file)
print()
print("Experiment Verdict:", verdict)
print("Control Conversion Rate:", f"{control_rate:.2%}")
print("Treatment Conversion Rate:", f"{treatment_rate:.2%}")
print("Absolute Effect:", f"{absolute_effect:.2%}")
print("Relative Effect:", f"{relative_effect:.2%}")
print("95% CI:", f"{ci_lower:.2%} to {ci_upper:.2%}")
print("P-Value:", round(p_value, 6))
print("SRM:", "Pass" if srm_pass else "Fail")
print(
    "Guardrails:",
    "Pass" if all_guardrails_pass else "Fail"
)
print()
print("Recommendation:")
print(recommendation)

PYTHON ANALYSIS COMPLETED SUCCESSFULLY
Output File: Python Analysis Result.xlsx

Experiment Verdict: INCONCLUSIVE
Control Conversion Rate: 67.89%
Treatment Conversion Rate: 69.83%
Absolute Effect: 1.94%
Relative Effect: 2.85%
95% CI: -10.17% to 14.05%
P-Value: 0.865256
SRM: Pass
Guardrails: Pass

Recommendation:
The evidence is not strong enough for a confident decision. Recommend collecting more data or running a better-powered experiment before making the final decision.
